In [ ]:
import os
import zipfile
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
from cnnClassifier import logger
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, get_size

In [ ]:
pwd

In [ ]:
os.chdir("../")

In [ ]:
pwd

In [ ]:
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [ ]:
@dataclass(frozen=True)
class KaggleToken:
    username: str
    key: str    

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        secrets_filepath = TOKEN_KEY_PATH,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.secrets = read_yaml(secrets_filepath)
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

    def get_data_ingestion_secrets(self) -> KaggleToken:
        secrets = self.secrets.kaggle_secrets

        get_data_ingestion_secrets = KaggleToken(
            username=secrets.username,
            key=secrets.key
        )

        return get_data_ingestion_secrets

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig, token: KaggleToken):
        self.config = config
        self.secrets = token
    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            os.environ['KAGGLE_USERNAME'] = self.secrets.username
            os.environ['KAGGLE_KEY'] = self.secrets.key
            url = self.config.source_URL            
            kaggle.api.authenticate()
            kaggle.api.dataset_download_files(url,
                                             path=self.config.unzip_dir,
                                             unzip=False)
            
            logger.info(f" Casting dataset is downloaded!")
        else:
            logger.info(f"File already exist and has a size of: {get_size(Path(self.config.local_data_file))}")


    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
            logger.info(f"Downloaded zipfile has been extracted")

In [ ]:
import kaggle 

In [ ]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion_secrets = config.get_data_ingestion_secrets()
    data_ingestion = DataIngestion(config=data_ingestion_config, token=data_ingestion_secrets)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e